# Circuit Breaker Pattern - Complete Guide for Beginners

## What is a Circuit Breaker?

**Real-world analogy**: Think of a circuit breaker in your home electricity panel.
- When there's too much electrical current (danger!), the circuit breaker "trips" (opens)
- It stops electricity flow to protect your appliances
- After some time, you can reset it to try again

**In Software**: A Circuit Breaker is a design pattern that:
- **Protects your application** from repeatedly calling a failing service
- **Prevents cascading failures** - when one service fails, it doesn't bring down your whole system
- **Saves resources** - stops wasting time/money on requests that will fail
- **Auto-recovers** - automatically tries again after some time


## Real-World Example

**Scenario**: Your e-commerce website depends on:
- Payment Service API
- Inventory Service API
- Shipping Service API

**Problem**: If Payment Service is down (maybe their database crashed):
- Every user checkout tries to call Payment Service
- Each call takes 30 seconds to timeout
- 1000 users → 1000 × 30 seconds = 30,000 seconds wasted!
- Your server gets overwhelmed and crashes too 😱

**Solution**: Circuit Breaker!
- After 5 failures, "open" the circuit (stop calling Payment Service)
- Immediately return error to user (no 30-second wait)
- After 60 seconds, try again (maybe service is fixed)
- If it works, "close" the circuit (normal operation resumes)


## Circuit Breaker States

A Circuit Breaker has **3 states**:

### 1. **CLOSED** (Normal State) ✅
- **Meaning**: Circuit is "closed" = current flows = requests go through normally
- **Behavior**: All requests are sent to the service
- **Tracking**: Counts failures (if service fails, increment failure count)
- **Transition**: If failures exceed threshold (e.g., 5 failures), move to OPEN state

### 2. **OPEN** (Failing State) 🔴
- **Meaning**: Circuit is "open" = current blocked = requests are rejected immediately
- **Behavior**: Requests are rejected immediately without calling the service
- **Timer**: Waits for a timeout period (e.g., 60 seconds)
- **Transition**: After timeout, move to HALF-OPEN state (test if service is back)

### 3. **HALF-OPEN** (Testing State) 🟡
- **Meaning**: Testing if service is recovered
- **Behavior**: Allows limited requests (e.g., 1-3 requests) to test the service
- **Success**: If requests succeed, move back to CLOSED (service is healthy)
- **Failure**: If requests fail, move back to OPEN (service still broken)

```
         CLOSED (Normal)
            ↓
    (5 failures detected)
            ↓
        OPEN (Blocking)
            ↓
    (60 seconds timeout)
            ↓
     HALF-OPEN (Testing)
       ↙        ↘
  SUCCESS     FAILURE
     ↓           ↓
  CLOSED       OPEN
```


## Step-by-Step: How It Works

### Example Timeline:

1. **Time 0:00** - Circuit is CLOSED
   - Request 1 → Service works ✅
   - Request 2 → Service works ✅

2. **Time 0:05** - Service starts failing
   - Request 3 → Service fails ❌ (failure count: 1)
   - Request 4 → Service fails ❌ (failure count: 2)
   - Request 5 → Service fails ❌ (failure count: 3)
   - Request 6 → Service fails ❌ (failure count: 4)
   - Request 7 → Service fails ❌ (failure count: 5) → **THRESHOLD REACHED!**

3. **Time 0:06** - Circuit OPENS 🔴
   - Request 8 → **Rejected immediately** (no service call, returns error instantly)
   - Request 9 → **Rejected immediately**
   - Request 10 → **Rejected immediately**
   - All future requests rejected until timeout...

4. **Time 1:06** - 60 seconds passed, Circuit goes HALF-OPEN 🟡
   - Request 11 → **Test request** sent to service
   - Service still down → **FAILURE** → Circuit goes back to OPEN

5. **Time 2:06** - Another 60 seconds passed, Circuit goes HALF-OPEN again
   - Request 12 → **Test request** sent to service
   - Service is back! → **SUCCESS** ✅ → Circuit goes to CLOSED

6. **Time 2:07** - Circuit is CLOSED again ✅
   - Request 13 → Service works ✅
   - Normal operation resumes


## Python Implementation

Let's build a simple Circuit Breaker from scratch!


In [1]:
# Step 1: Define the states using an Enum (enumeration)
# An Enum is like a set of named constants
from enum import Enum
import time

class CircuitState(Enum):
    """Three states of circuit breaker"""
    CLOSED = "CLOSED"      # Normal operation - requests go through
    OPEN = "OPEN"          # Service is failing - reject requests immediately
    HALF_OPEN = "HALF_OPEN" # Testing if service recovered
    
print("States available:")
for state in CircuitState:
    print(f"  - {state.name}: {state.value}")


States available:
  - CLOSED: CLOSED
  - OPEN: OPEN
  - HALF_OPEN: HALF_OPEN


In [2]:
# Step 2: Create the Circuit Breaker class
class CircuitBreaker:
    """
    Simple Circuit Breaker implementation
    
    Parameters:
    - failure_threshold: How many failures before opening circuit (default: 5)
    - timeout: How long to wait before trying again (default: 60 seconds)
    - half_open_max_calls: How many test calls in HALF_OPEN state (default: 3)
    """
    
    def __init__(self, failure_threshold=5, timeout=60, half_open_max_calls=3):
        # Configuration
        self.failure_threshold = failure_threshold  # Open circuit after 5 failures
        self.timeout = timeout  # Wait 60 seconds before testing again
        self.half_open_max_calls = half_open_max_calls  # Allow 3 test calls
        
        # State tracking
        self.state = CircuitState.CLOSED  # Start in CLOSED state
        self.failure_count = 0  # Count consecutive failures
        self.last_failure_time = None  # When did last failure happen?
        self.half_open_calls = 0  # How many calls made in HALF_OPEN state
        
    def call(self, func, *args, **kwargs):
        """
        Call a function through the circuit breaker
        
        Args:
            func: The function to call (e.g., API call)
            *args, **kwargs: Arguments to pass to the function
        
        Returns:
            Result from the function if successful
        
        Raises:
            CircuitBreakerOpenError: If circuit is OPEN
            Original exception: If function call fails
        """
        
        # Check if we should transition states
        self._check_and_transition_state()
        
        # Handle based on current state
        if self.state == CircuitState.OPEN:
            raise CircuitBreakerOpenError(
                f"Circuit breaker is OPEN. Service unavailable. "
                f"Will retry after {self.timeout} seconds."
            )
        
        elif self.state == CircuitState.HALF_OPEN:
            # In HALF_OPEN, we allow limited calls for testing
            if self.half_open_calls >= self.half_open_max_calls:
                raise CircuitBreakerOpenError(
                    "Circuit breaker is HALF_OPEN. Max test calls reached."
                )
            self.half_open_calls += 1
        
        # Try to call the function
        try:
            result = func(*args, **kwargs)
            # Success! Reset failure count and change state
            self._on_success()
            return result
            
        except Exception as e:
            # Failure! Increment count and update state
            self._on_failure()
            raise e  # Re-raise the original exception
    
    def _check_and_transition_state(self):
        """Check if we need to change state (OPEN → HALF_OPEN based on timeout)"""
        if self.state == CircuitState.OPEN:
            # Check if timeout period has passed
            if self.last_failure_time and (time.time() - self.last_failure_time) >= self.timeout:
                print(f"⏰ Timeout ({self.timeout}s) passed. Moving to HALF_OPEN state for testing...")
                self.state = CircuitState.HALF_OPEN
                self.half_open_calls = 0  # Reset test call counter
    
    def _on_success(self):
        """Handle successful call"""
        self.failure_count = 0  # Reset failure count
        
        if self.state == CircuitState.HALF_OPEN:
            print("✅ Service is back! Moving to CLOSED state.")
            self.state = CircuitState.CLOSED
        
    def _on_failure(self):
        """Handle failed call"""
        self.failure_count += 1
        self.last_failure_time = time.time()
        
        if self.state == CircuitState.CLOSED:
            print(f"❌ Failure {self.failure_count}/{self.failure_threshold}")
            if self.failure_count >= self.failure_threshold:
                print(f"🔴 Circuit breaker OPENED after {self.failure_count} failures!")
                self.state = CircuitState.OPEN
                
        elif self.state == CircuitState.HALF_OPEN:
            print(f"❌ Test call failed. Moving back to OPEN state.")
            self.state = CircuitState.OPEN
            self.half_open_calls = 0
    
    def get_state(self):
        """Get current state of circuit breaker"""
        return self.state.value
    
    def reset(self):
        """Manually reset circuit breaker to CLOSED state"""
        self.state = CircuitState.CLOSED
        self.failure_count = 0
        self.last_failure_time = None
        self.half_open_calls = 0
        print("🔄 Circuit breaker manually reset to CLOSED state.")


# Custom exception for when circuit is open
class CircuitBreakerOpenError(Exception):
    """Exception raised when circuit breaker is OPEN"""
    pass

print("✅ CircuitBreaker class created!")


✅ CircuitBreaker class created!


## Example 1: Simulating a Failing Service

Let's create a mock service that fails randomly to test our circuit breaker!


In [3]:
import random

# Simulate a service that sometimes fails
class UnreliableService:
    def __init__(self, failure_rate=0.7):
        """
        failure_rate: Probability of failure (0.0 = never fails, 1.0 = always fails)
        """
        self.failure_rate = failure_rate
        self.call_count = 0
    
    def call_service(self):
        """Simulates calling an external API"""
        self.call_count += 1
        
        # Simulate network delay
        time.sleep(0.1)  # 100ms delay
        
        # Randomly succeed or fail
        if random.random() < self.failure_rate:
            raise Exception(f"Service error: Database connection failed (call #{self.call_count})")
        
        return f"Service response: Success! (call #{self.call_count})"

# Create service that fails 70% of the time
service = UnreliableService(failure_rate=0.7)

# Create circuit breaker
cb = CircuitBreaker(failure_threshold=3, timeout=5)  # Open after 3 failures, wait 5 seconds

print("Service and Circuit Breaker ready!")
print(f"Initial state: {cb.get_state()}")


Service and Circuit Breaker ready!
Initial state: CLOSED


In [4]:
# Test the circuit breaker
print("\n=== Testing Circuit Breaker ===\n")

for i in range(10):
    print(f"\nRequest {i+1}:")
    print(f"  Current state: {cb.get_state()}")
    
    try:
        # Call service through circuit breaker
        result = cb.call(service.call_service)
        print(f"  ✅ {result}")
        
    except CircuitBreakerOpenError as e:
        print(f"  🔴 {e}")
        
    except Exception as e:
        print(f"  ❌ {e}")
    
    time.sleep(0.5)  # Small delay between requests



=== Testing Circuit Breaker ===


Request 1:
  Current state: CLOSED
❌ Failure 1/3
  ❌ Service error: Database connection failed (call #1)

Request 2:
  Current state: CLOSED
❌ Failure 2/3
  ❌ Service error: Database connection failed (call #2)

Request 3:
  Current state: CLOSED
❌ Failure 3/3
🔴 Circuit breaker OPENED after 3 failures!
  ❌ Service error: Database connection failed (call #3)

Request 4:
  Current state: OPEN
  🔴 Circuit breaker is OPEN. Service unavailable. Will retry after 5 seconds.

Request 5:
  Current state: OPEN
  🔴 Circuit breaker is OPEN. Service unavailable. Will retry after 5 seconds.

Request 6:
  Current state: OPEN
  🔴 Circuit breaker is OPEN. Service unavailable. Will retry after 5 seconds.

Request 7:
  Current state: OPEN
  🔴 Circuit breaker is OPEN. Service unavailable. Will retry after 5 seconds.

Request 8:
  Current state: OPEN
  🔴 Circuit breaker is OPEN. Service unavailable. Will retry after 5 seconds.

Request 9:
  Current state: OPEN
  🔴 Circuit

## Example 2: Real API Call Simulation

Let's simulate what happens with a real payment API that goes down:


In [5]:
# Simulate payment service
class PaymentService:
    def __init__(self, is_healthy=True):
        self.is_healthy = is_healthy
    
    def process_payment(self, amount):
        """Simulate processing a payment"""
        if not self.is_healthy:
            raise Exception("Payment service is down! Database connection failed.")
        
        # Simulate processing time
        time.sleep(0.2)
        return f"Payment of ${amount} processed successfully!"
    
    def set_health(self, healthy):
        """Manually set service health (for testing)"""
        self.is_healthy = healthy
        status = "healthy" if healthy else "down"
        print(f"🏥 Payment service status changed to: {status}")

# Create payment service (starts healthy, then we'll make it fail)
payment_service = PaymentService(is_healthy=True)

# Create circuit breaker for payment service
payment_cb = CircuitBreaker(failure_threshold=3, timeout=3)

print("=== Payment Service with Circuit Breaker ===\n")


=== Payment Service with Circuit Breaker ===



In [6]:
# Scenario: Service is healthy, then goes down, then recovers

print("Phase 1: Service is healthy ✅")
print("-" * 50)
for i in range(3):
    try:
        result = payment_cb.call(payment_service.process_payment, 100)
        print(f"Request {i+1}: {result}")
    except Exception as e:
        print(f"Request {i+1}: ❌ {e}")

print("\nPhase 2: Service goes down! 😱")
print("-" * 50)
payment_service.set_health(False)

for i in range(5):
    try:
        result = payment_cb.call(payment_service.process_payment, 100)
        print(f"Request {i+1}: {result}")
    except CircuitBreakerOpenError as e:
        print(f"Request {i+1}: 🔴 Circuit OPEN - {e}")
    except Exception as e:
        print(f"Request {i+1}: ❌ {e}")

print("\n⏳ Waiting for timeout period...")
time.sleep(4)  # Wait for circuit to go HALF_OPEN

print("\nPhase 3: Testing if service recovered...")
print("-" * 50)
try:
    result = payment_cb.call(payment_service.process_payment, 100)
    print(f"Test request: {result}")
except Exception as e:
    print(f"Test request: ❌ {e}")

print("\nPhase 4: Service is back! 🎉")
print("-" * 50)
payment_service.set_health(True)

time.sleep(4)  # Wait for next timeout

try:
    result = payment_cb.call(payment_service.process_payment, 100)
    print(f"Request after recovery: {result}")
except Exception as e:
    print(f"Request after recovery: ❌ {e}")


Phase 1: Service is healthy ✅
--------------------------------------------------
Request 1: Payment of $100 processed successfully!
Request 2: Payment of $100 processed successfully!
Request 3: Payment of $100 processed successfully!

Phase 2: Service goes down! 😱
--------------------------------------------------
🏥 Payment service status changed to: down
❌ Failure 1/3
Request 1: ❌ Payment service is down! Database connection failed.
❌ Failure 2/3
Request 2: ❌ Payment service is down! Database connection failed.
❌ Failure 3/3
🔴 Circuit breaker OPENED after 3 failures!
Request 3: ❌ Payment service is down! Database connection failed.
Request 4: 🔴 Circuit OPEN - Circuit breaker is OPEN. Service unavailable. Will retry after 3 seconds.
Request 5: 🔴 Circuit OPEN - Circuit breaker is OPEN. Service unavailable. Will retry after 3 seconds.

⏳ Waiting for timeout period...

Phase 3: Testing if service recovered...
--------------------------------------------------
⏰ Timeout (3s) passed. Moving

## Example 3: Comparing With and Without Circuit Breaker

Let's see the difference in performance:


In [7]:
import time

class SlowFailingService:
    def __init__(self):
        self.is_down = False
    
    def call(self):
        if self.is_down:
            time.sleep(3)  # Simulate 3-second timeout
            raise Exception("Service timeout")
        return "Success"

# Service that goes down after first call
service = SlowFailingService()
service.call()  # First call succeeds
service.is_down = True  # Then it goes down

print("=== WITHOUT Circuit Breaker ===")
print("Making 10 requests to failing service...")
start_time = time.time()

for i in range(10):
    try:
        service.call()
    except:
        pass  # Ignore errors

time_without_cb = time.time() - start_time
print(f"⏱️  Time taken: {time_without_cb:.2f} seconds (each request waited 3 seconds!)")

print("\n=== WITH Circuit Breaker ===")
service2 = SlowFailingService()
service2.call()  # First call succeeds
service2.is_down = True

cb2 = CircuitBreaker(failure_threshold=1, timeout=60)  # Open after 1 failure

print("Making 10 requests to failing service...")
start_time = time.time()

for i in range(10):
    try:
        cb2.call(service2.call)
    except CircuitBreakerOpenError:
        pass  # Circuit open - immediate rejection
    except:
        pass  # First failure

time_with_cb = time.time() - start_time
print(f"⏱️  Time taken: {time_with_cb:.2f} seconds (circuit opened, rejections are instant!)")

print(f"\n💰 Time saved: {time_without_cb - time_with_cb:.2f} seconds")
print(f"🚀 Speed improvement: {time_without_cb / time_with_cb:.1f}x faster")


=== WITHOUT Circuit Breaker ===
Making 10 requests to failing service...
⏱️  Time taken: 30.03 seconds (each request waited 3 seconds!)

=== WITH Circuit Breaker ===
Making 10 requests to failing service...
❌ Failure 1/1
🔴 Circuit breaker OPENED after 1 failures!
⏱️  Time taken: 3.00 seconds (circuit opened, rejections are instant!)

💰 Time saved: 27.03 seconds
🚀 Speed improvement: 10.0x faster


## Key Concepts Summary

### 1. **Why Use Circuit Breaker?**
- Prevents cascading failures
- Saves time and resources (no waiting for timeouts)
- Better user experience (fast error responses)
- Automatic recovery when service comes back

### 2. **When to Use?**
- Calling external APIs (payment, email, SMS services)
- Database connections
- Microservices communication
- Any service that can fail and has slow timeouts

### 3. **Configuration Parameters**
- **failure_threshold**: How many failures before opening (e.g., 5)
- **timeout**: How long to wait before testing again (e.g., 60 seconds)
- **half_open_max_calls**: How many test calls in HALF_OPEN (e.g., 3)

### 4. **Important Points**
- Circuit breaker doesn't fix the service - it protects YOUR application
- Different services should have different circuit breakers
- Monitor circuit breaker states to know which services are down
- Can be manually reset if needed

## Real-World Libraries

Instead of building from scratch, you can use:
- **Python**: `circuitbreaker` library, `pybreaker`
- **Java**: `Resilience4j` (mentioned in your notes), `Hystrix`
- **Node.js**: `opossum`, `brakes`
- **Go**: `gobreaker`

## Practice Exercise

Try modifying the code to:
1. Add success threshold (need X successes in HALF_OPEN to go to CLOSED)
2. Add metrics tracking (count how many requests were rejected)
3. Add different timeout strategies (exponential backoff)

---

**Congratulations!** 🎉 You now understand Circuit Breaker pattern!

This is a fundamental pattern used in microservices architecture and distributed systems.
